# 00 - Setup & Workflow

**Sources:** Bootcamp BP1 (dev environment, venvs, project skeleton, VCS) and Summer School Part I
("Working with Python": workflow, IPython vs CLI, editors, IDEs, notebooks).

**Goal:** be able to create an isolated environment, install dependencies, run code in the three
usual ways (REPL, script, notebook) and lay out a small project the way a team would.

## 1. Which Python am I running?

Python is a *specification*; the default implementation is **CPython** (others: PyPy, Jython,
IronPython). Always check which interpreter a notebook/terminal is really using - most
"it works on my machine" problems are interpreter/venv mix-ups.

In [1]:
import platform
import sys

print("Version     :", platform.python_version())
print("Implementation:", platform.python_implementation())
print("Executable  :", sys.executable)
print("Prefix      :", sys.prefix)
print("Base prefix :", sys.base_prefix)

Version     : 3.12.3
Implementation: CPython
Executable  : C:\Users\leire\Projects\Python-Refresher\.venv\Scripts\python.exe
Prefix      : C:\Users\leire\Projects\Python-Refresher\.venv
Base prefix : C:\Users\leire\AppData\Local\Programs\Python\Python312


In [2]:
# Inside a virtual environment, sys.prefix != sys.base_prefix
in_venv = sys.prefix != sys.base_prefix
print("Running inside a virtual environment:", in_venv)

Running inside a virtual environment: True


## 2. Virtual environments (BP1)

`pip install x` installs *globally* unless a venv is active, so two projects needing different
versions of the same package clash. A virtual environment is just a folder with its own
interpreter + `site-packages`.

```powershell
# Windows (PowerShell) - what this repo uses
python -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install -r requirements.txt
deactivate
```

```bash
# macOS / Linux
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
```

Tips:
* Use `python -m pip ...` (not bare `pip`) so you are sure pip belongs to *this* interpreter.
* Never commit the venv folder - commit `requirements.txt` instead (it is in `.gitignore`).
* Alternatives with extra features (declarative deps, lock files): `virtualenv`, `pipenv`,
  `poetry`, `conda`, and today also `uv`.
* The bootcamp used Docker dev containers instead of venvs: the container *is* the isolated
  environment, and dependencies are installed inside it from `requirements.txt`.

## 3. `requirements.txt` and `requirements-dev.txt`

* `requirements.txt` - third-party packages needed **to run** the project.
* `requirements-dev.txt` - extra tools needed **to develop** it (formatters, linters, tests).
  In production only the first one is installed.

```text
numpy>=1.26        # lower bound
pandas==2.2.*      # compatible-release pin
```

`python -m pip freeze > requirements.lock.txt` records the exact versions currently installed.

## 4. Three ways of running Python

| Mode | How | Good for |
|---|---|---|
| REPL / IPython | `python` or `ipython` | trying one-liners, exploring an API (`obj.<TAB>`, `obj?`) |
| Script | `python script.py arg1 arg2` | anything repeatable; the real "program" |
| Notebook | `jupyter lab` | exploration, teaching, plots, notes next to code |

Rule of thumb: **explore in a notebook, then move the stable code into `.py` modules** and
import it back (this is exactly the workflow in the `projects/` folder).

### IPython / Jupyter niceties (from the Summer School slides)

In [3]:
# `?` shows the docstring, `??` the source (works in IPython / Jupyter)
sorted?

In [4]:
# Line magics
%timeit sum(range(1000))

7.02 μs ± 83.7 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [5]:
# The who / whos magics list the variables in the session; run executes a script in this namespace
a = 10
b = "text"
%whos

Variable   Type      Data/Info
------------------------------
a          int       10
b          str       text
in_venv    bool      True
platform   module    <module 'platform' from '<...>on312\\Lib\\platform.py'>
sys        module    <module 'sys' (built-in)>


In [6]:
# Tab-completion discovers methods: type `r.` then <TAB> in a code cell.
# Without a UI you can do the same with dir():
r = [3, 1, 2]
print([m for m in dir(r) if not m.startswith("_")])

['append', 'clear', 'copy', 'count', 'extend', 'index', 'insert', 'pop', 'remove', 'reverse', 'sort']


## 5. Editors and IDEs

* **PyCharm** (bootcamp): refactor menu (rename / extract / inline / move / change signature),
  *Search Everywhere* (double Shift), context actions (Alt+Enter), Ctrl+click to jump to a
  definition, run/debug configurations, built-in VCS and Docker integration.
* **VS Code / Cursor**: same ideas through the Python + Jupyter extensions; select the `.venv`
  interpreter with *Python: Select Interpreter*.
* **Spyder**: Matlab-like, scientist-oriented IDE (used in the Summer School).

Useful features whatever you use: autocompletion, snippets, tab-to-spaces, a debugger with
breakpoints, and a formatter on save.

## 6. Project skeleton (BP1)

Most professional Python projects look like this:

```text
.
├── Makefile                # shortcuts: make format / static / test / test_coverage
├── README.md
├── docker/                 # Dockerfile (prod) + dev.dockerfile (dev)
├── docker-compose.yaml
├── requirements.txt
├── requirements-dev.txt
├── src/                    # the code
└── test/                   # the tests
```

The Makefile typically wraps:

| Target | Runs | Purpose |
|---|---|---|
| `format` / `format_check` | `black`, `isort` | enforce PEP 8 style automatically |
| `static` | `flake8`, `mypy` | lint + static type checking |
| `test` | `pytest` / `unittest` | run the tests |
| `test_coverage` | `pytest --cov` | how much of the code the tests execute |

On Windows there is usually no `make`; just run the commands directly (see notebook 11), or put
them in a small `tasks.ps1`.

## 7. Git cheat-sheet (BP1 / BP8 outline)

```bash
git config --global user.name  "Your Name"
git config --global user.email "you@example.com"

git clone <url>                  # copy a remote repo
git status                       # what changed?
git add <file>   |  git add .   # stage
git commit -m "Short imperative message"
git push  /  git pull

git switch -c feature/my-idea    # new branch
git merge main                   # bring main into the current branch
git rebase main                  # replay my commits on top of main (linear history)
git log --oneline --graph --all
```

Workflow used by teams: **fork/branch -> commit -> push -> merge request (pull request)**.
The MR runs automated *checks*: pipeline (CI) status, code review, merge conflicts.
Use **SSH keys** instead of passwords for remotes (`ssh-keygen -t ed25519`, add the public key to
GitHub/GitLab).

*Practice:* every notebook / project in this repo = one small branch + commit message.

## 8. Mini-checklist (do this once)

- [ ] `python -m venv .venv` and install `requirements.txt`
- [ ] Register the kernel: `python -m ipykernel install --user --name python-refresher`
- [ ] Run `python -m pytest codility -q` - you should see many *skipped* tests (not implemented yet)
- [ ] Open `jupyter lab` from the repo root and pick the `.venv` kernel